# EDA 001.03 — Parsing Dates

**Kaggle Data Cleaning Course — Lesson 3**

Dates in real-world data come in many inconsistent formats. This notebook covers:

1. Common **date formats** and why they cause problems
2. Parsing dates with `pd.to_datetime()`
3. Handling **mixed formats** and errors
4. Extracting useful features (year, month, day of week)
5. **Visualizing** temporal patterns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1 — Load the Dataset

In [ ]:
df = pd.read_csv("data/eda_001_03_events.csv")
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
# Check current dtypes — dates are loaded as strings!
df.dtypes

## 2 — The Problem: Dates as Strings

Notice how `event_date` and `registration_deadline` are `object` (string) types.

Let's look at the variety of formats in our data:

In [ ]:
# Show diverse date formats in the data
print("Sample event_date values:")
for i in range(10):
    print(f"  Row {i}: {df['event_date'].iloc[i]}")

print("\nSample registration_deadline values:")
for i in range(10):
    print(f"  Row {i}: {df['registration_deadline'].iloc[i]}")

## 3 — Parsing Dates with `pd.to_datetime()`

### 3a — Automatic parsing

Pandas can often **infer** the format automatically. The key parameters:
- `format=` — specify exact format (fastest)
- `infer_datetime_format=True` — let pandas guess (slower but flexible)
- `errors='coerce'` — invalid dates become `NaT` instead of raising errors
- `dayfirst=True` — for European-style dd/mm/yyyy

In [ ]:
# Try automatic parsing with mixed formats
df["event_date_parsed"] = pd.to_datetime(df["event_date"], format="mixed", dayfirst=False)
df["reg_deadline_parsed"] = pd.to_datetime(df["registration_deadline"], format="mixed", dayfirst=False)

print("Parsed dtypes:")
print(f"  event_date_parsed      : {df['event_date_parsed'].dtype}")
print(f"  reg_deadline_parsed    : {df['reg_deadline_parsed'].dtype}")

# Check for any parsing failures (NaT)
print(f"\nNaT in event_date_parsed  : {df['event_date_parsed'].isna().sum()}")
print(f"NaT in reg_deadline_parsed: {df['reg_deadline_parsed'].isna().sum()}")

In [ ]:
# Compare original string vs parsed datetime
df[["event_date", "event_date_parsed", "registration_deadline", "reg_deadline_parsed"]].head(10)

### 3b — Using `errors='coerce'` to handle bad dates

In [ ]:
# Demonstrate coercing bad dates
sample_dates = pd.Series(["2020-01-15", "not-a-date", "02/30/2021", "2019-12-25"])
parsed = pd.to_datetime(sample_dates, errors="coerce")

for orig, result in zip(sample_dates, parsed):
    print(f"  '{orig}' → {result}")

## 4 — Extracting Date Components

Once dates are proper `datetime64`, we can extract rich temporal features using the `.dt` accessor:

In [ ]:
# Extract components from the parsed event_date
df["year"] = df["event_date_parsed"].dt.year
df["month"] = df["event_date_parsed"].dt.month
df["day"] = df["event_date_parsed"].dt.day
df["day_of_week"] = df["event_date_parsed"].dt.day_name()
df["quarter"] = df["event_date_parsed"].dt.quarter
df["week_of_year"] = df["event_date_parsed"].dt.isocalendar().week.astype(int)

# Compute lead time: days between registration deadline and event
df["lead_time_days"] = (df["event_date_parsed"] - df["reg_deadline_parsed"]).dt.days

df[["event_date_parsed", "year", "month", "day", "day_of_week", "quarter", "lead_time_days"]].head(10)

## 5 — Visualizing Temporal Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Events per year
df["year"].value_counts().sort_index().plot.bar(ax=axes[0, 0], color="steelblue", edgecolor="white")
axes[0, 0].set_title("Events per Year")
axes[0, 0].set_xlabel("Year")
axes[0, 0].set_ylabel("Count")

# Events per month
month_counts = df["month"].value_counts().sort_index()
month_counts.index = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                       "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"][:len(month_counts)]
month_counts.plot.bar(ax=axes[0, 1], color="coral", edgecolor="white")
axes[0, 1].set_title("Events per Month")
axes[0, 1].set_xlabel("Month")

# Events per day of week
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_counts = df["day_of_week"].value_counts().reindex(day_order)
day_counts.plot.bar(ax=axes[1, 0], color="mediumseagreen", edgecolor="white")
axes[1, 0].set_title("Events per Day of Week")
axes[1, 0].set_xlabel("Day")

# Lead time distribution
axes[1, 1].hist(df["lead_time_days"].dropna(), bins=20, color="mediumpurple", edgecolor="white")
axes[1, 1].set_title("Registration Lead Time (days before event)")
axes[1, 1].set_xlabel("Days")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Timeline plot: events over time by type
fig, ax = plt.subplots(figsize=(16, 5))

event_types = df["event_name"].unique()
colors = sns.color_palette("Set2", len(event_types))

for event_type, color in zip(event_types, colors):
    mask = df["event_name"] == event_type
    ax.scatter(df.loc[mask, "event_date_parsed"], df.loc[mask, "attendees"],
               label=event_type, alpha=0.7, s=50, color=color)

ax.set_title("Events Timeline: Attendance Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Attendees")
ax.legend(title="Event Type")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: average attendees by month and event type
pivot = df.pivot_table(values="attendees", index="event_name",
                        columns="month", aggfunc="mean")

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax)
ax.set_title("Average Attendees by Month and Event Type")
ax.set_xlabel("Month")
ax.set_ylabel("Event Type")
plt.tight_layout()
plt.show()

## 6 — Common Date Format Reference

| Code | Meaning | Example |
|------|---------|----------|
| `%Y` | 4-digit year | 2024 |
| `%y` | 2-digit year | 24 |
| `%m` | Month (01–12) | 03 |
| `%B` | Full month name | March |
| `%b` | Abbreviated month | Mar |
| `%d` | Day (01–31) | 15 |
| `%H` | Hour (00–23) | 14 |
| `%M` | Minute (00–59) | 30 |
| `%S` | Second (00–59) | 45 |

In [ ]:
# Demonstrate explicit format parsing (much faster for large datasets)
from datetime import datetime

examples = [
    ("2024-03-15",      "%Y-%m-%d"),
    ("03/15/2024",      "%m/%d/%Y"),
    ("15-Mar-2024",     "%d-%b-%Y"),
    ("March 15, 2024",  "%B %d, %Y"),
    ("15.03.2024",      "%d.%m.%Y"),
]

print("Explicit format parsing examples:")
for date_str, fmt in examples:
    parsed = datetime.strptime(date_str, fmt)
    print(f"  '{date_str}' with format '{fmt}' → {parsed.date()}")

## 7 — Key Takeaways

- **Always convert date strings** to `datetime64` early in your pipeline
- Use `format="mixed"` or `errors="coerce"` for messy real-world data
- Specifying an exact `format=` string is **10–100x faster** than auto-inference
- Extract **year, month, day_of_week** as features for ML models
- Compute **time deltas** (differences between dates) for derived features
- Watch out for **timezone** issues: use `pd.to_datetime(..., utc=True)` when needed